In [3]:
import asyncio
import json
import re
import inspect

from ollama import AsyncClient

import config as config
from tools_registry import AVAILABLE_TOOLS
from context_manager import ContextAwareClient, MessageCompressor



In [ ]:
from llm import (
    MockToolCall, MockFunction, extract_fallback_tools, execute_tool, 
    validate_plan, run_unguided_loop,
    get_tools_with_signatures, 
    # review_and_enhance_plan,
    execute_plan,
    plan_execution
)

In [ ]:
async def review_and_enhance_plan(client: AsyncClient, user_prompt: str, draft_plan: list[dict]) -> list[dict]:
    """
    An independent reviewer turn (no conversation context shared) that audits, corrects,
    and refines the generated draft execution plan for mathematical and logical consistency.
    """
    print("[REVIEWER] 🔍 Initiating isolated plan auditing loop...\n")

    # Define the Reviewer's System Guidelines
    reviewer_system_prompt = (
        "You are an expert Planning Auditor and Optimizer. Your task is to review a draft step-by-step execution plan "
        "designed to satisfy a user request. You must ensure the plan is logical, sequentially correct, avoids redundancies, "
        "and maps parameter dependencies correctly using step placeholder tags (e.g. <result_of_step_N>).\n\n"
        "Strict Review Constraints:\n"
        "- The only allowed tools and their exact callable signatures are:\n"
        "{available_tools}\n"
        "- Check for and ELIMINATE redundant filler steps (e.g., adding 0, multiplying/dividing by 1, or creating intermediate placeholder steps that perform no useful work).\n"
        "- Double-check the user's mathematical intent against the tool selection! If they ask to 'multiply' or 'product', ensure you use 'multiply_numbers' and NOT 'add_numbers'. If they ask to 'add' or 'sum', use 'add_numbers'.\n"
        "- Verify that placeholder tokens correctly point to a prior step's outputs.\n"
        "- Ensure math operations follow standard precedence rules.\n"
        "- Minimize the total number of steps. If a plan can be resolved in 3 clean steps instead of 5 bloated steps, modify and consolidate it.\n"
        "- Output ONLY the final validated valid JSON array of steps. Absolutely no conversational filler, markdown code fences, or headers."
    )

    review_user_message = (
        f"Original User Request: '{user_prompt}'\n\n"
        f"Draft Plan to audit:\n{json.dumps(draft_plan, indent=2)}\n\n"
        f"Analyze the draft plan, apply any corrections or enhancements, and output the audited plan JSON:"
    )

    reviewer_messages = [
        {"role": "system", "content": reviewer_system_prompt.format(
            available_tools=get_tools_with_signatures())},
        {"role": "user", "content": review_user_message}
    ]

    try:
        # Execute turning call (using clean, independent client execution)
        response = await client.chat(
            model=config.MODEL_NAME,
            messages=reviewer_messages,
            options={**config.LLM_OPTIONS, "temperature": 0.0}
        )

        raw = response.message.content or ""

        # Use our robust array parser to extract JSON list regardless of surrounding conversational preamble
        raw = re.sub(r"\x60{3}(?:json)?", "", raw).strip().rstrip("\x60").strip()

        plan = json.loads(raw)

        if plan and validate_plan(plan):
            print(
                f"[REVIEWER] ✅ Plan audit complete! Enhanced plan contains {len(plan)} step(s):")
            for step in plan:
                print(f"   Step {step.get('step')}: {step.get('description')} "
                      f"→ {step.get('tool')}({step.get('args')})")
            print()
            return plan
        else:
            print(
                "[REVIEWER] ⚠️ Reviewed plan parsed but failed structural validation. Falling back to draft plan.")
    except Exception as e:
        print(
            f"[REVIEWER] ❌ Review phase failed due to parse error: {str(e)}. Falling back to draft plan.")

    return draft_plan

In [19]:
async def run_conversation(user_prompt: str):
    client = AsyncClient(host=config.OLLAMA_HOST)

    print(f"\n[---] Starting Agent for prompt: '{user_prompt}'\n")

    # --- PHASE 1: DRAFT PLAN ---
    draft_plan = await plan_execution(client, user_prompt)

    # --- PHASE 2: REVIEW & ENHANCE (NEW) ---
    plan = []
    if draft_plan:
        plan = await review_and_enhance_plan(client, user_prompt, draft_plan)

    # --- PHASE 3: EXECUTE ---
    if plan:
        await execute_plan(client, plan, user_prompt)
    else:
        print(
            "[---] Planning/Auditing failed completely. Falling back to unguided agentic loop.\n")
        await run_unguided_loop(client, user_prompt)

In [20]:
prompt = "Calculate number of times RCB won in IPL and multiply that number with number of times MI won"
await  run_conversation(prompt)


[---] Starting Agent for prompt: 'Calculate number of times RCB won in IPL and multiply that number with number of times MI won'

[PLANNER] 🧠 Generating draft execution plan...

[
  {"step": 1, "tool": "web_search_ddg", "args": {"query": "RCB IPL wins count", "max_results": 1}, "description": "Search for the number of times RCB won in IPL"},
  {"step": 2, "tool": "web_search_ddg", "args": {"query": "MI IPL wins count", "max_results": 1}, "description": "Search for the number of times MI won in IPL"},
  {"step": 3, "tool": "multiply_numbers", "args": {"a": "<result_of_step_1>", "b": "<result_of_step_2>"}, "description": "Multiply RCB wins with MI wins"}
]
[PLANNER] 📝 Draft Plan with 3 step(s) created:
   Step 1: Search for the number of times RCB won in IPL → web_search_ddg({'query': 'RCB IPL wins count', 'max_results': 1})
   Step 2: Search for the number of times MI won in IPL → web_search_ddg({'query': 'MI IPL wins count', 'max_results': 1})
   Step 3: Multiply RCB wins with MI wins